In [1]:
import pandas as pd
import psycopg2
from pathlib import Path


In [3]:

# ============================================================
# CONFIGURATION
# ============================================================

RAW_DIR = Path("../data/raw")

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "insuraflow",
    "user": "postgres",
    "password": "postgres"
}

SCHEMA = "insurance"

LOAD_ORDER = [
    "locations",
    "customers",
    "agents",
    "vehicles",
    "policies",
    "policy_coverage",
    "premium_payments",
    "garages",
    "hospitals",
    "claims",
    "claim_assessments",
    "claim_documents",
    "claim_payments"
]


# ============================================================
# DATE COLUMNS
# ============================================================

DATE_COLUMNS = {
    "customers": [
        "date_of_birth",
        "kyc_verified_date",
        "registration_date"
    ],
    "agents": [
        "joining_date"
    ],
    "vehicles": [
        "registration_date"
    ],
    "policies": [
        "issue_date",
        "start_date",
        "end_date",
        "renewal_date"
    ],
    "premium_payments": [
        "payment_date"
    ],
    "claims": [
        "claim_date",
        "incident_date"
    ],
    "claim_assessments": [
        "assessment_date"
    ],
    "claim_documents": [
        "uploaded_date"
    ],
    "claim_payments": [
        "payment_date"
    ]
}


# ============================================================
# CONNECT
# ============================================================

try:

    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = False

    print("PostgreSQL connection successful.\n")

except Exception as e:

    raise Exception(
        f"PostgreSQL connection failed: {e}"
    )


# ============================================================
# LOAD TABLES
# ============================================================

try:

    for table_name in LOAD_ORDER:

        file_path = RAW_DIR / f"{table_name}.csv"

        if not file_path.exists():

            raise FileNotFoundError(
                f"Missing file: {file_path}"
            )


        # ----------------------------------------------------
        # Read CSV
        # ----------------------------------------------------

        df = pd.read_csv(
            file_path
        )


        # ----------------------------------------------------
        # Convert date columns
        # ----------------------------------------------------

        if table_name in DATE_COLUMNS:

            for column in DATE_COLUMNS[table_name]:

                if column in df.columns:

                    df[column] = pd.to_datetime(
                        df[column],
                        errors="coerce"
                    )


        # ----------------------------------------------------
        # Convert pandas missing values to Python None
        # ----------------------------------------------------

        df = df.astype(object)

        df = df.where(
            pd.notna(df),
            None
        )


        # ----------------------------------------------------
        # Cursor
        # ----------------------------------------------------

        cursor = conn.cursor()


        # ----------------------------------------------------
        # Clear existing table
        # ----------------------------------------------------

        cursor.execute(
            f'''
            TRUNCATE TABLE
                "{SCHEMA}"."{table_name}"
            RESTART IDENTITY CASCADE;
            '''
        )


        # ----------------------------------------------------
        # Columns
        # ----------------------------------------------------

        columns = list(
            df.columns
        )

        column_sql = ", ".join(
            f'"{column}"'
            for column in columns
        )

        placeholders = ", ".join(
            ["%s"] * len(columns)
        )


        insert_sql = f'''
            INSERT INTO
                "{SCHEMA}"."{table_name}"
                ({column_sql})
            VALUES
                ({placeholders})
        '''


        # ----------------------------------------------------
        # Convert DataFrame rows
        # ----------------------------------------------------

        rows = [
            tuple(row)
            for row in df.itertuples(
                index=False,
                name=None
            )
        ]


        # ----------------------------------------------------
        # Insert
        # ----------------------------------------------------

        cursor.executemany(
            insert_sql,
            rows
        )

        conn.commit()


        print(
            f"{table_name:20} | "
            f"{len(df):6,} rows loaded"
        )


        cursor.close()


except Exception as e:

    conn.rollback()

    print("\nLOAD FAILED")
    print(f"Error: {e}")

    raise


finally:

    conn.close()

    print("\nPostgreSQL connection closed.")


# ============================================================
# FINAL MESSAGE
# ============================================================

print("\n======================================")
print("RAW DATA LOAD COMPLETED SUCCESSFULLY")
print("======================================")

PostgreSQL connection successful.

locations            |     20 rows loaded
customers            |    500 rows loaded
agents               |     30 rows loaded
vehicles             |    500 rows loaded
policies             |    750 rows loaded
policy_coverage      |  2,249 rows loaded
premium_payments     |  1,211 rows loaded
garages              |     30 rows loaded
hospitals            |     20 rows loaded
claims               |    300 rows loaded
claim_assessments    |    257 rows loaded
claim_documents      |    601 rows loaded
claim_payments       |     68 rows loaded

PostgreSQL connection closed.

RAW DATA LOAD COMPLETED SUCCESSFULLY
